<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/saneamento/Esgoto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Esgotamento sanitário — população atendida com rede coletora

Motor de coleta e publicação da série `esgoto-rede-coletora-total`.

- **Fonte:** SNIS — Série Histórica, *Informações e indicadores municipais consolidados*
  (`https://app4.cidades.gov.br/serieHistorica/consolidadoMunicipio/index`), informação
  **ES001 — População total atendida com esgotamento sanitário**, de 1995 a 2022; e
  **Sinisa** — planilhas do módulo Esgotamento Sanitário, ano de referência 2023
  (`SINISA_ESGOTO_Planilhas_2023_v2.zip`, consolidado UF/MR/Brasil), para 2023.
- **Denominador:** população residente do Brasil do **IBGE**, informação **POP_TOT** da
  própria Série Histórica (soma dos municípios de todo o país; reproduz o total censitário
  nos anos de Censo — 169.799.170 em 2000 e 190.755.799 em 2010; em 2022, 203.080.756 é a
  estimativa do ano, 0,009% acima do Censo de 203.062.512). Não é a população da amostra do
  SNIS/Sinisa.
- **Definição:** percentual da população residente do Brasil atendida por rede coletora de
  esgoto. **Não é esgoto tratado:** mede quem tem coleta, não quem tem tratamento.
- **Fórmula:** `percentual = 100 × população total atendida com rede coletora ÷ população
  residente do Brasil (IBGE)`, por ano.
- **Quem não reporta entra como não atendido.** O numerador soma apenas a população que os
  prestadores declararam atender; município sem declaração não entra. Por isso a série é um
  **piso** da cobertura real.
- **Unidade:** percentual (`unidade.tipo: percentual`, `simbolo: %`, `casas: 1`);
  `eixo: percentual`, `medida: nivel`, `granularidade: anual`.
- **Período publicado:** `1995-01-01` a `2023-01-01` (29 pontos).
- **Arquivo gerado:** `indicadores/saneamento/esgoto-rede-coletora-total.json`.

## Por que a série termina em 2023

O SNIS fez a última coleta em 2023 (ano de referência 2022) e o Sinisa começou em 2023 — os
dois sistemas **não têm ano de sobreposição**. O critério do projeto era emendar apenas se a
diferença no ano de sobreposição fosse ≤ 0,1 p.p.; sem sobreposição, o teste possível é o
degrau 2022 → 2023 com o mesmo denominador IBGE, que é de **−0,71 p.p.** (55,54% → 54,83%).
Acima da tolerância, a série para em 2023 e 2024 fica de fora: o Sinisa 2024 existe, mas
acrescentá-lo seria emendar sem conseguir medir a quebra.

O degrau não é queda real de atendimento. O Sinisa 2023 cobre 186,5 milhões de habitantes
(municípios que declararam), contra 201,4 milhões na base de água do SNIS 2022. O índice
oficial do Sinisa (59,7%) usa essa população menor como denominador — por isso ele é 4,9
p.p. maior que o número nacional publicado aqui.

## O que este número é — e o que ele não é

- **É a população com rede coletora**, urbana e rural somadas (ES001 e GTE0001 + GTE0002).
  Não é só a urbana: quem mora em área rural atendida entra no numerador.
- **Não é esgoto tratado.** O município pode coletar e lançar o esgoto sem tratamento; a
  série não mede tratamento nem poluição evitada.
- **O denominador é o país inteiro, não a amostra.** É o que separa esta série do índice
  oficial (IN056/IES0001): o SNIS 2022 publica 56,0% sobre a população dos municípios com
  água (201,4 milhões); aqui a mesma população atendida (112,8 milhões) é dividida pelos
  203,1 milhões do Censo 2022, dando 55,54%.
- **Os anos recentes são revisados.** O SNIS revisa a base e o IBGE revisa a população; o
  ponto de 2022 ilustra os dois efeitos ao mesmo tempo (Censo 2022 no denominador e a queda
  de resposta do último ano do SNIS). Reexecutar este notebook muda valores se a fonte for
  revisada — aí é revisão de fonte, registrada em `docs/metodologia.md`.


## Coleta

Para cada ano de 1995 a 2022 o notebook faz uma busca na grade da Série Histórica com
`tipo_info = 1` (todos os municípios do Brasil) e pede três informações no mesmo recorte:
**ES001** (população atendida, numerador), **POP_TOT** (população residente do IBGE,
denominador) e **G12A** (população dos municípios com água, que reproduz o denominador do
índice oficial).

O total vem do rodapé (`somatorio`), que o servidor calcula sobre **todas** as linhas da
consulta. A base municipal consolidada tem uma linha por município — 5.570 em 2022 — e a soma
das linhas é idêntica ao totalizador, então o mesmo habitante não é contado duas vezes.
A consulta é repetida em falha de rede, com espera crescente: a API do SNIS falha de forma
intermitente.


In [1]:
import io
import json
import os
import random
import string
import time
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "saneamento"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("SANEAMENTO_SAIDA", PASTA))


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


BASE_SNIS = "https://app4.cidades.gov.br/serieHistorica/consolidadoMunicipio"
PAGINA_SNIS = "https://app4.cidades.gov.br/serieHistorica/"
URL_SINISA = (
    "https://www.gov.br/cidades/pt-br/acesso-a-informacao/acoes-e-programas/saneamento/"
    "sinisa/resultados-sinisa/SINISA_ESGOTO_Planilhas_2023_v2.zip"
)
PRIMEIRO_ANO = 1995
ULTIMO_ANO_SNIS = 2022  # última coleta do SNIS (ano de referência 2022)
ANO_SINISA = 2023
COD_ES001 = 38738    # ES001 - População total atendida com esgotamento sanitário
COD_POP_TOT = 39153  # POP_TOT - População total do município do ano de referência (IBGE)
COD_G12A = 39207     # G12A - População total residente dos municípios com abastecimento de água
NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
RID = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"


In [2]:
def postar(sessao, rota, dados, tentativas=5):
    """POST na API do SNIS com repetição e espera crescente (a falha é intermitente)."""
    for tentativa in range(tentativas):
        try:
            resposta = sessao.post(f"{BASE_SNIS}/{rota}", data=dados, timeout=120)
            resposta.raise_for_status()
            return resposta.json()
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


def formulario(ano):
    """Formulário da grade: todos os municípios, um ano, as três informações do recorte."""
    return (
        "ShMunicipios[tipo_info][]=1"
        f"&ShMunicipios[ano_ref][]={ano}"
        "&ShMunicipios[cod_fam_info][]=1&ShMunicipios[cod_fam_info][]=5"
        f"&ShMunicipios[fk_glossario][]={COD_POP_TOT}"
        f"&ShMunicipios[fk_glossario][]={COD_G12A}"
        f"&ShMunicipios[fk_glossario][]={COD_ES001}"
    )


def somatorio_do_ano(sessao, ano):
    """ES001, POP_TOT e G12A somados sobre todas as linhas do ano (uma por município)."""
    identificador = "".join(random.choices(string.ascii_lowercase + string.digits, k=12))
    resposta = postar(
        sessao,
        "getGridConfig",
        {"data": formulario(ano), "starting": 1, "search-id": identificador},
    )
    for _ in range(10):
        if resposta.get("isFinished"):
            total = resposta.get("somatorio") or {}
            if any(total.get(codigo) is None for codigo in ("es001", "pop_tot", "g12a")):
                raise ValueError(f"{ano}: totalizador incompleto ({total})")
            return float(total["es001"]), float(total["pop_tot"]), float(total["g12a"])
        # A matriz é montada em etapas no servidor; a busca é recuperada pelo identificador.
        resposta = postar(sessao, "getGridConfig", {"search-id": identificador})
    raise RuntimeError(f"{ano}: a matriz de resultados não fechou")


sessao = requests.Session()
postar(sessao, "index", {"ajax": 1})

atendida_snis = {}
populacao_snis = {}
populacao_agua_snis = {}
for ano in range(PRIMEIRO_ANO, ULTIMO_ANO_SNIS + 1):
    atendida_snis[ano], populacao_snis[ano], populacao_agua_snis[ano] = (
        somatorio_do_ano(sessao, ano)
    )
    print(
        f"{ano}: {atendida_snis[ano]:>14,.0f} atendidos / "
        f"{populacao_snis[ano]:>14,.0f} habitantes"
    )


1995:      8,677,000 atendidos /    155,822,440 habitantes


1996:     30,494,368 atendidos /    157,070,163 habitantes


1997:     35,757,542 atendidos /    159,636,413 habitantes


1998:     39,848,218 atendidos /    161,790,311 habitantes


1999:     47,457,778 atendidos /    163,947,554 habitantes


2000:     52,021,038 atendidos /    169,799,170 habitantes


... (18 outputs left out)

2019:    110,300,342 atendidos /    210,147,125 habitantes


2020:    114,615,022 atendidos /    211,755,692 habitantes


2021:    117,313,123 atendidos /    213,317,639 habitantes


2022:    112,795,436 atendidos /    203,080,756 habitantes


## Sinisa 2023

O Sinisa é a continuação do SNIS a partir de 2023. A planilha consolidada UF/MR/Brasil traz
a linha `TOTAL NACIONAL` com a população residente e a população atendida (urbana + rural).
O índice oficial IES0001 é `(GTE0001 + GTE0002) / DFE0001 × 100`; o notebook o recalcula da
planilha para provar que leu o recorte certo — o valor tem de bater com os 59,7% do
relatório do Sinisa (referência 2023).


In [3]:
def baixar(url, tentativas=4, timeout=180):
    """GET do .zip com repetição; o servidor do gov.br trunca downloads com frequência."""
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, timeout=timeout)
            resposta.raise_for_status()
            if not resposta.content.startswith(b"PK"):
                raise ValueError(f"{url} não devolveu um .zip")
            with zipfile.ZipFile(io.BytesIO(resposta.content)) as pacote:
                if pacote.testzip() is not None:
                    raise ValueError(f"{url}: arquivo corrompido")
            return resposta.content
        except (requests.RequestException, ValueError, zipfile.BadZipFile):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


def folhas(conteudo):
    """Mapa nome da folha -> caminho do XML dentro do .xlsx."""
    with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
        livro = ET.fromstring(pacote.read("xl/workbook.xml"))
        relacoes = ET.fromstring(pacote.read("xl/_rels/workbook.xml.rels"))
        alvos = {relacao.get("Id"): relacao.get("Target") for relacao in relacoes}
        return {
            folha.get("name"): "xl/" + alvos[folha.get(RID)].lstrip("/").replace("xl/", "")
            for folha in livro.iter(NS + "sheet")
        }


def linhas(conteudo, caminho):
    """Linhas da folha como dicionários coluna -> texto (zip de XML, sem openpyxl)."""
    with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
        compartilhadas = []
        if "xl/sharedStrings.xml" in pacote.namelist():
            for item in ET.fromstring(pacote.read("xl/sharedStrings.xml")):
                compartilhadas.append("".join(t.text or "" for t in item.iter(NS + "t")))
        folha = ET.fromstring(pacote.read(caminho))
        saida = []
        for linha in folha.iter(NS + "row"):
            valores = {}
            for celula in linha.iter(NS + "c"):
                coluna = "".join(ch for ch in celula.get("r") if ch.isalpha())
                bruto = celula.find(NS + "v")
                if bruto is None:
                    embutido = celula.find(NS + "is")
                    valor = (
                        "".join(t.text or "" for t in embutido.iter(NS + "t"))
                        if embutido is not None
                        else None
                    )
                elif celula.get("t") == "s":
                    valor = compartilhadas[int(bruto.text)]
                else:
                    valor = bruto.text
                valores[coluna] = valor
            saida.append(valores)
        return saida


def planilha_consolidada(conteudo):
    """A planilha consolidada UF/MR/Brasil dentro do .zip de planilhas do Sinisa."""
    with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
        nome = next(
            item
            for item in pacote.namelist()
            if item.endswith(".xlsx")
            and "Consolidado" in item
            and ("Gestão T" in item or "Gestao T" in item)
        )
        return pacote.read(nome)


def total_nacional(conteudo):
    """População residente e população atendida do Brasil na planilha consolidada.

    A 2023 não tem a coluna de população total atendida (só urbana e rural); a 2024 tem
    `GTE0023`. As duas formas são aceitas e conferidas contra o índice oficial.
    """
    nome = next(n for n in folhas(conteudo) if n.startswith("Gestão Técnica"))
    grade = linhas(conteudo, folhas(conteudo)[nome])
    cabecalho = next(
        linha
        for linha in grade
        if any((valor or "").startswith("População Total Residente") for valor in linha.values())
    )
    colunas = {valor.strip(): coluna for coluna, valor in cabecalho.items() if valor}
    total = next(
        (coluna for valor, coluna in colunas.items() if valor.startswith("População total atendida")),
        None,
    )
    urbana = colunas["População urbana atendida com rede de esgotamento sanitário"]
    rural = colunas["População rural atendida com rede de esgotamento sanitário"]
    linha = next(l for l in grade if l.get("A") in ("TOTAL NACIONAL", "Brasil"))
    atendida = (
        float(linha[total])
        if total
        else float(linha[urbana] or 0) + float(linha[rural] or 0)
    )
    return float(linha[colunas["População Total Residente"]]), atendida


conteudo_sinisa = baixar(URL_SINISA)
populacao_sinisa, atendida_sinisa = total_nacional(planilha_consolidada(conteudo_sinisa))
indice_sinisa = 100.0 * atendida_sinisa / populacao_sinisa
print(
    f"Sinisa {ANO_SINISA}: {atendida_sinisa:,.0f} atendidos / "
    f"{populacao_sinisa:,.0f} habitantes = {indice_sinisa:.2f}% (índice da amostra)"
)


Sinisa 2023: 111,344,129 atendidos / 186,511,726 habitantes = 59.70% (índice da amostra)


## Cálculo e emenda

A fórmula é aplicada ano a ano, com o denominador nacional. Para 2023 não existe população
residente do IBGE publicada na base do Censo 2022: a publicação do DOU de 2023 trouxe a
população de 2022, e o Sinisa estimou a população de 2023 a partir do Censo por metodologia
simplificada. O notebook usa a estimativa de 2022 do próprio SNIS (203.080.756), que difere
do Censo (203.062.512) em 0,009% — 0,005 p.p. na taxa. É a única população nacional
disponível para 2023 na mesma base dos demais anos.

O degrau da emenda é medido com o **mesmo denominador** dos dois lados: 2022 pelo SNIS e
2023 pelo Sinisa. Como o SNIS não publica 2023, não há ano de sobreposição para o teste de
0,1 p.p. do projeto; o degrau é o teste possível, e ele é maior que a tolerância.


In [4]:
# Fórmula: percentual = 100 × população total atendida ÷ população residente do Brasil.
taxa = {
    ano: 100.0 * atendida_snis[ano] / populacao_snis[ano]
    for ano in range(PRIMEIRO_ANO, ULTIMO_ANO_SNIS + 1)
}

# 2023: a população de 2022 (203.080.756) como denominador — o IBGE não publicou estimativa
# de 2023 na base do Censo, e o Censo 2022 (203.062.512) difere em 0,009% (0,005 p.p.).
populacao_2023 = populacao_snis[2022]
taxa[ANO_SINISA] = 100.0 * atendida_sinisa / populacao_2023

# Índice oficial do SNIS (IN056): mesma população atendida sobre os municípios com água.
indice_oficial = {
    ano: 100.0 * atendida_snis[ano] / populacao_agua_snis[ano]
    for ano in range(PRIMEIRO_ANO, ULTIMO_ANO_SNIS + 1)
}

degrau = taxa[ANO_SINISA] - taxa[ULTIMO_ANO_SNIS]
print(f"emenda 2022 (SNIS) -> 2023 (Sinisa), mesmo denominador: {degrau:+.2f} p.p.")
print(f"  SNIS 2022:   {taxa[2022]:.2f}%  (índice oficial IN056: {indice_oficial[2022]:.2f}%)")
print(f"  Sinisa 2023: {taxa[2023]:.2f}%  (índice oficial IES0001: {indice_sinisa:.2f}%)")
assert abs(degrau) > 0.1, (
    "o degrau da emenda ficou dentro da tolerância: reavalie publicar 2024"
)
print("degrau acima de 0,1 p.p.: a série termina em 2023 (2024 fica de fora).")
print()
print(f"{'ano':>4} {'atendida':>14} {'população':>14} {'taxa':>6}")
for ano in sorted(taxa):
    atendida = atendida_snis[ano] if ano <= ULTIMO_ANO_SNIS else atendida_sinisa
    populacao = populacao_snis[ano] if ano <= ULTIMO_ANO_SNIS else populacao_2023
    print(f"{ano:>4} {atendida:>14,.0f} {populacao:>14,.0f} {taxa[ano]:>6.2f}")


emenda 2022 (SNIS) -> 2023 (Sinisa), mesmo denominador: -0.71 p.p.
  SNIS 2022:   55.54%  (índice oficial IN056: 56.00%)
  Sinisa 2023: 54.83%  (índice oficial IES0001: 59.70%)
degrau acima de 0,1 p.p.: a série termina em 2023 (2024 fica de fora).

 ano       atendida      população   taxa
1995      8,677,000    155,822,440   5.57
1996     30,494,368    157,070,163  19.41
1997     35,757,542    159,636,413  22.40
1998     39,848,218    161,790,311  24.63
1999     47,457,778    163,947,554  28.95
2000     52,021,038    169,799,170  30.64
2001     53,182,692    172,383,193  30.85
2002     55,470,201    174,632,960  31.76
2003     57,446,253    176,871,437  32.48
2004     61,110,974    181,576,184  33.66
2005     64,991,723    184,184,264  35.29
2006     69,121,175    186,770,562  37.01
2007     70,681,109    183,987,291  38.42
2008     74,441,261    189,612,814  39.26
2009     78,606,622    191,506,729  41.05
2010     83,838,228    190,755,799  43.95
2011     87,439,926    192,379,287  4

## Escrita do JSON do contrato

O arquivo é gravado inteiro (nada de merge com o que está publicado), com `versao`, `id`,
`unidade`, `eixo`, `medida`, `granularidade`, `fonte`, `urlFonte` e `dados` no formato
`{"data": "YYYY-MM-DD", "valor": ...}`. O valor vai **sem arredondamento**; `unidade.casas`
é só exibição.


In [5]:
FONTE = (
    "SNIS - Série Histórica (ES001, 1995-2022) e Sinisa (2023): população total atendida "
    "com rede coletora de esgoto; percentual calculado pelo site sobre a população "
    "residente do Brasil do IBGE"
)

serie = {
    "versao": 1,
    "id": "esgoto-rede-coletora-total",
    "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 1},
    "eixo": "percentual",
    "medida": "nivel",
    "granularidade": "anual",
    "fonte": FONTE,
    "urlFonte": PAGINA_SNIS,
    "dados": [
        {"data": f"{ano}-01-01", "valor": taxa[ano]}
        for ano in range(PRIMEIRO_ANO, ANO_SINISA + 1)
    ],
}

DESTINO.mkdir(parents=True, exist_ok=True)
arquivo = DESTINO / "esgoto-rede-coletora-total.json"
with arquivo.open("w", encoding="utf-8") as saida:
    json.dump(serie, saida, ensure_ascii=False, indent=2)
    saida.write("\n")
print(f"{len(serie['dados'])} pontos escritos em {relativo(arquivo)}")


29 pontos escritos em indicadores/saneamento/esgoto-rede-coletora-total.json


In [6]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}

assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
assert serie["versao"] == 1
assert serie["id"] == "esgoto-rede-coletora-total"
assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 1}
assert serie["eixo"] == "percentual"
assert serie["medida"] == "nivel"
assert serie["granularidade"] == "anual"
assert isinstance(serie["fonte"], str) and serie["fonte"]
assert serie["urlFonte"].startswith("https://")

dados = serie["dados"]
assert dados, "série vazia"
datas = [ponto["data"] for ponto in dados]
anos_lista = [int(data[:4]) for data in datas]

# C15 anual: todo ponto em AAAA-01-01, um por ano, ordem ascendente.
assert datas[0] == "1995-01-01", datas[0]
assert datas[-1] == "2023-01-01", datas[-1]
assert all(data.endswith("-01-01") for data in datas), "dia diferente de 01"
assert all(a < b for a, b in zip(datas, datas[1:])), "datas fora de ordem"
assert anos_lista == list(range(1995, 2024)), "ano faltando ou repetido"
assert len(dados) == 29, len(dados)

import math

for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

# Fórmula: cada ponto é 100 × atendida ÷ população do mesmo ano.
for ponto in dados:
    ano = int(ponto["data"][:4])
    if ano <= ULTIMO_ANO_SNIS:
        esperado = 100.0 * atendida_snis[ano] / populacao_snis[ano]
    else:
        esperado = 100.0 * atendida_sinisa / populacao_2023
    assert ponto["valor"] == esperado, (ponto, esperado)

with (DESTINO / "esgoto-rede-coletora-total.json").open(encoding="utf-8") as aberto:
    assert json.load(aberto) == serie, "arquivo gravado difere do objeto"


def valor(ano):
    return {int(p["data"][:4]): p["valor"] for p in dados}[ano]


# Conferência com o que a fonte publica no mesmo recorte.
# SNIS, Diagnóstico Temático AE 2023 (dados de 2022): 112,8 milhões atendidos, IN056 = 56,0%.
assert atendida_snis[2022] == 112_795_436, atendida_snis[2022]
assert abs(indice_oficial[2022] - 56.0) < 0.05, indice_oficial[2022]
# Sinisa 2024 (dados de 2023): IES0001 = 59,7% (111,3 milhões).
assert abs(indice_sinisa - 59.7) < 0.05, indice_sinisa

# Âncoras do denominador: os anos de Censo têm de reproduzir o total do IBGE (em 2022, a
# estimativa do ano, 0,009% acima do Censo, é a base do SNIS).
assert populacao_snis[2000] == 169_799_170, populacao_snis[2000]
assert populacao_snis[2010] == 190_755_799, populacao_snis[2010]
assert populacao_snis[2022] == 203_080_756, populacao_snis[2022]

# Âncoras de valor: falham numa revisão de fonte. Se acontecer, confira a base e registre
# a revisão em docs/metodologia.md antes de mexer no número.
for ano, esperado in [
    (1995, 5.5685),
    (2000, 30.6368),
    (2010, 43.9506),
    (2015, 48.6230),
    (2020, 54.1261),
    (2022, 55.5422),
    (2023, 54.8275),
]:
    assert abs(valor(ano) - esperado) < 1e-4, (ano, valor(ano), esperado)

# 2023 fica abaixo do índice oficial (59,7%) e ligeiramente abaixo da faixa de 55-62%
# estimada no plano, porque o denominador é o país inteiro e a amostra do Sinisa cobre
# 186,5 dos 203,1 milhões de habitantes. O valor é o piso nacional, não o índice da amostra.
assert 54.5 < valor(2023) < 55.1, valor(2023)

print(
    f"ok: {len(dados)} pontos ({datas[0]} a {datas[-1]}); fórmula, dia 01, ordem, "
    "cobertura, contrato e conferências contra as fontes."
)


ok: 29 pontos (1995-01-01 a 2023-01-01); fórmula, dia 01, ordem, cobertura, contrato e conferências contra as fontes.
